This notebook deals with the part 2 of the project.
- I did the initial EDA , Data cleaning and Data Imputation of the Median_Income data set 
- Saved the cleaned median data as a parquet file
- Next I cleaned the Crime Data set using the EDA results we got in the "crime_data_eda.ipynb" and saved it as .csv file
- At the end I merged the parquet and csv files to make a merged data set on which we will do our later parts of the project

In [103]:
import pandas as pd ,numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, hour, month, year, when, count, desc, dayofmonth, to_date

# Median Income Data Set:

In [76]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark with the Postgres driver
spark = SparkSession.builder \
    .appName("LA-Crime-Research") \
    .getOrCreate()
    
# Load the Median Income Data 
df_income = spark.read.csv("CS226_project/Median_Income_and_AMI_(census_tract).csv", header=True, inferSchema=True)
df_income.show(5)


+----------+-------------+----------------------+---------------+----------------+----------------------+---------------------+----------+--------------------+--------------------+--------+------------------+----------------+
|     tract|med_hh_income|med_hh_income_universe|   ami_category|below_med_income|below_60pct_med_income|below_moderate_income|  sup_dist|                 csa|                 spa|ESRI_OID|       Shape__Area|   Shape__Length|
+----------+-------------+----------------------+---------------+----------------+----------------------+---------------------+----------+--------------------+--------------------+--------+------------------+----------------+
|6037101110|        84091|                  1558|     Low Income|             Yes|                    No|                  Yes|District 5|Los Angeles - Tuj...|SPA 2 - San Fernando|    4842|1.23298062148438E7|14765.6490038758|
|6037101122|        99583|                  1407|     Low Income|              No|              

26/02/18 19:25:19 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [77]:
df_income.printSchema()

root
 |-- tract: long (nullable = true)
 |-- med_hh_income: integer (nullable = true)
 |-- med_hh_income_universe: integer (nullable = true)
 |-- ami_category: string (nullable = true)
 |-- below_med_income: string (nullable = true)
 |-- below_60pct_med_income: string (nullable = true)
 |-- below_moderate_income: string (nullable = true)
 |-- sup_dist: string (nullable = true)
 |-- csa: string (nullable = true)
 |-- spa: string (nullable = true)
 |-- ESRI_OID: integer (nullable = true)
 |-- Shape__Area: double (nullable = true)
 |-- Shape__Length: double (nullable = true)



## 2. EDA + Data Cleaning:


Used Pandas for this purpose

#### 1. Overview:

In [78]:

data = pd.read_csv("CS226_project/Median_Income_and_AMI_(census_tract).csv")
data.head(5)

# 1. Data set Overview:
print("Dataset Overview")
print(f"Total Rows: {data.shape[0]}, Total Columns: {data.shape[1]}")

# 2. Calculate the number of nulls and the percentage:
print("% of Null values per column")
null_counts = data.isnull().sum()
null_percentages = (null_counts / len(data)) * 100
# Create a summary table
null_summary = pd.DataFrame({
    'Null Count': null_counts,
    'Percentage (%)': null_percentages
})
print(null_summary)
      

Dataset Overview
Total Rows: 2495, Total Columns: 13
% of Null values per column
                        Null Count  Percentage (%)
tract                            0        0.000000
med_hh_income                   42        1.683367
med_hh_income_universe           0        0.000000
ami_category                    42        1.683367
below_med_income                42        1.683367
below_60pct_med_income          42        1.683367
below_moderate_income           42        1.683367
sup_dist                         0        0.000000
csa                              0        0.000000
spa                              0        0.000000
ESRI_OID                         0        0.000000
Shape__Area                      0        0.000000
Shape__Length                    0        0.000000


**Observation**

In this dataset, exactly 42 census tracts (1.68%) are missing income-specific data.
- Geographic/ID (tract, spa, ESRI_OID)  00.00%
- Income Metrics (med_hh_income, ami_category)   1.68%

#### 2. Socio Economic Insights:

In [79]:
# Income stats by region:
avg_income_spa = data.groupby('spa')['med_hh_income'].mean().sort_values(ascending=False)
print("\n--- Average Income by Service Planning Area (SPA) ---")
print(avg_income_spa)

# Distribution of AMI categories:
print("\n--- AMI Category Distribution ---")
print(data['ami_category'].value_counts())


--- Average Income by Service Planning Area (SPA) ---
spa
SPA 5 - West               131263.522222
SPA 2 - San Fernando       102003.381387
SPA 8 - South Bay          100073.457534
SPA 3 - San Gabriel         99429.498721
SPA 7 - East                86813.947735
SPA 1 - Antelope Valley     83880.122222
SPA 4 - Metro               77333.547550
SPA 6 - South               61994.008163
Name: med_hh_income, dtype: float64

--- AMI Category Distribution ---
ami_category
Low Income               1094
Very Low Income           638
Above Moderate Income     527
Moderate Income           104
Extremely Low Income       85
Acutely Low Income          5
Name: count, dtype: int64


**Observations**

- **Integrity**: Approximately 1.68% of census tracts (42 out of 2,495) are missing income data and were removed during cleaning.

- **Wealth Gap**: The West (SPA 5) is the wealthiest region with an average income of ~$131,263, while the South (SPA 6) is the lowest at ~$61,994.

- **Dominant Group**: "Low Income" is the most frequent tract classification in Los Angeles.

#### 3. Data Cleaning:

In [80]:
# 1. Deal with missing values:

# Calculate mean income for the 'Low Income' group
low_income_mean = data[data['ami_category'] == 'Low Income']['med_hh_income'].mean()
print(f"Mean income for 'Low Income' group: {low_income_mean}")

# Perform the replacement
data_imputed = data.copy()
data_imputed['med_hh_income'] = data_imputed['med_hh_income'].fillna(low_income_mean)

# Verify nulls are gone in med_hh_income
new_nulls = data_imputed['med_hh_income'].isnull().sum()
print(f"New null count for med_hh_income: {new_nulls}")

# Show a few rows that were previously null (if possible)
# Finding indices of original nulls
null_indices = data[data['med_hh_income'].isnull()].index
print("\nSample of imputed values:")
print(data_imputed.loc[null_indices, ['tract', 'med_hh_income']].head())

Mean income for 'Low Income' group: 88984.63345521023
New null count for med_hh_income: 0

Sample of imputed values:
          tract  med_hh_income
36   6037115103   88984.633455
461  6037190303   88984.633455
569  6037204410   88984.633455
598  6037207307   88984.633455
938  6037265301   88984.633455


In [81]:
# 2. Standardize 'tract' as a string to prevent joining errors
data_imputed['tract'] = data_imputed['tract'].astype(str)

# 3. Standardize text data
data_imputed['spa'] = data_imputed['spa'].str.strip()


## 3. Save the cleaned File :

In [89]:
# Save cleaned local copy
data_imputed.to_csv('Median_Income_Cleaned.csv', index=False)
print("\nCleaned local file saved as: Median_Income_Cleaned.csv")


Cleaned local file saved as: Median_Income_Cleaned.csv


# LA Crime Data Set:

### Data Cleaning and Saved:
EDA is done in the seperate notebook

In [83]:
from pyspark.sql.functions import col, to_date, year, month, dayofmonth, when, trim

# 1. Load Data
df_crime = spark.read.csv("CS226_project/Crime_Data.csv", header=True, inferSchema=True)

# 2. Rename Columns (Standardize for SQL and Joining)
# This removes spaces like "Crm Cd Desc" -> "Crm_Cd_Desc"
for col_name in df_crime.columns:
    df_crime = df_crime.withColumnRenamed(col_name, col_name.replace(" ", "_"))

# 3. Standardize Dates
# Converts 'DATE_OCC' (MM/dd/yyyy hh:mm:ss a) to a proper Date type
df_crime = df_crime.withColumn("Date_Occured", to_date(col("DATE_OCC"), "MM/dd/yyyy hh:mm:ss a")) \
                   .withColumn("Date_Reported", to_date(col("Date_Rptd"), "MM/dd/yyyy hh:mm:ss a"))

# 4. Extract Temporal Features
df_crime = df_crime.withColumn("Occured_Year", year("Date_Occured")) \
                   .withColumn("Occured_Month", month("Date_Occured")) \
                   .withColumn("Occured_Day", dayofmonth("Date_Occured")) \
                   .withColumn("Hour", (col("TIME_OCC") / 100).cast("int"))

# 5. Geospatial & Demographic Cleaning
# Filter out 0/0 coordinates and invalid victim ages
df_crime_clean = df_crime.filter((col("LAT") != 0) & (col("LON") != 0)) \
                         .filter(col("Vict_Age") > 0)

# 6. Feature Engineering: Weapon Flag
df_crime_clean = df_crime_clean.withColumn(
    "Weapon_Used", 
    when(col("Weapon_Used_Cd").isNotNull(), "Yes").otherwise("No")
)



In [84]:
# 7. Save locally to your project folder
# as .csv
df_crime_clean.repartition(1).write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save("crime_cleaned_local.csv")

# as .parquet
# Saves to the local folder in your VS Code workspace
df_crime_clean.write.mode("overwrite").parquet("crime_cleaned_local.parquet")

26/02/18 19:25:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/18 19:25:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/18 19:25:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/02/18 19:25:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/02/18 19:25:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/02/18 19:25:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/02/18 19:25:24 WARN MemoryManager: Total allocation exceeds 95.

### Verify 

In [85]:
# 1. Load the cleaned data back
df_verify = spark.read.csv("crime_cleaned_local.csv", header=True, inferSchema=True)

# 2. Check the row count (Compare this to your original count)
print(f"Total Records in Cleaned File: {df_verify.count()}")

# 3. Verify column names (Spaces should be replaced by underscores)
print("Cleaned Columns:", df_verify.columns)

# 4. Check that coordinate filtering worked (Should return 0)
invalid_coords = df_verify.filter((col("LAT") == 0) | (col("LON") == 0)).count()
print(f"Records with invalid coordinates: {invalid_coords}")

# 5. Show a sample of the cleaned data
df_verify.select("DR_NO", "Date_Occured", "AREA_NAME", "Crm_Cd_Desc").show(5)

Total Records in Cleaned File: 733940
Cleaned Columns: ['DR_NO', 'Date_Rptd', 'DATE_OCC', 'TIME_OCC', 'AREA', 'AREA_NAME', 'Rpt_Dist_No', 'Part_1-2', 'Crm_Cd', 'Crm_Cd_Desc', 'Mocodes', 'Vict_Age', 'Vict_Sex', 'Vict_Descent', 'Premis_Cd', 'Premis_Desc', 'Weapon_Used_Cd', 'Weapon_Desc', 'Status', 'Status_Desc', 'Crm_Cd_1', 'Crm_Cd_2', 'Crm_Cd_3', 'Crm_Cd_4', 'LOCATION', 'Cross_Street', 'LAT', 'LON', 'Date_Occured', 'Date_Reported', 'Occured_Year', 'Occured_Month', 'Occured_Day', 'Hour', 'Weapon_Used']
Records with invalid coordinates: 0
+---------+------------+-----------+--------------------+
|    DR_NO|Date_Occured|  AREA_NAME|         Crm_Cd_Desc|
+---------+------------+-----------+--------------------+
|211507896|  2020-11-07|N Hollywood|   THEFT OF IDENTITY|
|201516622|  2020-10-18|N Hollywood|ASSAULT WITH DEAD...|
|240913563|  2020-10-30|   Van Nuys|   THEFT OF IDENTITY|
|210704711|  2020-12-24|   Wilshire|THEFT FROM MOTOR ...|
|201418201|  2020-09-29|    Pacific|THEFT FROM MOTOR

In [86]:
# 1. Read from your local project directory
df_local_parquet = spark.read.parquet("crime_cleaned_local.parquet")

# 2.  Check total count to ensure it matches your original dataset
print(f"Total Rows: {df_local_parquet.count()}")
df_local_parquet.printSchema()

Total Rows: 733940
root
 |-- DR_NO: integer (nullable = true)
 |-- Date_Rptd: string (nullable = true)
 |-- DATE_OCC: string (nullable = true)
 |-- TIME_OCC: integer (nullable = true)
 |-- AREA: integer (nullable = true)
 |-- AREA_NAME: string (nullable = true)
 |-- Rpt_Dist_No: integer (nullable = true)
 |-- Part_1-2: integer (nullable = true)
 |-- Crm_Cd: integer (nullable = true)
 |-- Crm_Cd_Desc: string (nullable = true)
 |-- Mocodes: string (nullable = true)
 |-- Vict_Age: integer (nullable = true)
 |-- Vict_Sex: string (nullable = true)
 |-- Vict_Descent: string (nullable = true)
 |-- Premis_Cd: integer (nullable = true)
 |-- Premis_Desc: string (nullable = true)
 |-- Weapon_Used_Cd: integer (nullable = true)
 |-- Weapon_Desc: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Status_Desc: string (nullable = true)
 |-- Crm_Cd_1: integer (nullable = true)
 |-- Crm_Cd_2: integer (nullable = true)
 |-- Crm_Cd_3: integer (nullable = true)
 |-- Crm_Cd_4: integer (null

In [87]:
# 3. Verify column names (Spaces should be replaced by underscores)
print("Cleaned Columns:", df_local_parquet.columns)

# 4. Check that coordinate filtering worked (Should return 0)
invalid_coords = df_local_parquet.filter((col("LAT") == 0) | (col("LON") == 0)).count()
print(f"Records with invalid coordinates: {invalid_coords}")

# 5. Show a sample of the cleaned data
df_local_parquet.select("DR_NO", "Date_Occured", "AREA_NAME", "Crm_Cd_Desc").show(5)

Cleaned Columns: ['DR_NO', 'Date_Rptd', 'DATE_OCC', 'TIME_OCC', 'AREA', 'AREA_NAME', 'Rpt_Dist_No', 'Part_1-2', 'Crm_Cd', 'Crm_Cd_Desc', 'Mocodes', 'Vict_Age', 'Vict_Sex', 'Vict_Descent', 'Premis_Cd', 'Premis_Desc', 'Weapon_Used_Cd', 'Weapon_Desc', 'Status', 'Status_Desc', 'Crm_Cd_1', 'Crm_Cd_2', 'Crm_Cd_3', 'Crm_Cd_4', 'LOCATION', 'Cross_Street', 'LAT', 'LON', 'Date_Occured', 'Date_Reported', 'Occured_Year', 'Occured_Month', 'Occured_Day', 'Hour', 'Weapon_Used']
Records with invalid coordinates: 0
+---------+------------+-----------+--------------------+
|    DR_NO|Date_Occured|  AREA_NAME|         Crm_Cd_Desc|
+---------+------------+-----------+--------------------+
|200617036|  2020-10-21|  Hollywood|            BURGLARY|
|201226392|  2020-12-14|77th Street|ASSAULT WITH DEAD...|
|201006499|  2020-02-24|West Valley|THEFT FROM MOTOR ...|
|201006614|  2020-02-28|West Valley|ASSAULT WITH DEAD...|
|200913158|  2020-08-05|   Van Nuys|       BIKE - STOLEN|
+---------+------------+--------

# Merging the Two Cleaned Data Sets:

To combine your two cleaned datasets (Crime and Income) into a single .csv file, we need a common key to link them. Since the Crime Data is incident-based (Latitude/Longitude) and the Income Data is geography-based (Census Tract), the standard "common part of schema" for our research is the Census Tract or the Service Planning Area (SPA).

**The Merge Strategy: Mapping Areas to Income**

Since the raw crime data doesn't contain the tract ID, we will join them using the Service Planning Area (SPA). This allows you to associate every crime incident with the average income and economic category of its broader region.

In [ ]:
# 1. Start the Session if not running already:
#spark = SparkSession.builder.appName("LA-Crime-Income-Join").getOrCreate()

# 2. Load the Cleaned Datasets:
df_crime = spark.read.parquet("crime_cleaned_local.parquet")
df_income = spark.read.csv("Median_Income_Cleaned.csv",header=True)

In [100]:
# 3. Create a Mapping Bridge
# In LA, LAPD Areas correspond to specific SPAs. 
# This 'Common Schema' allows the join to work.
df_crime_with_spa = df_crime.withColumn("spa", 
    when(col("AREA_NAME").isin("Central", "Rampart", "Newton", "Hollenbeck"), "SPA 4 - Metro")
    .when(col("AREA_NAME").isin("77th Street", "Southwest", "Southeast"), "SPA 6 - South")
    .when(col("AREA_NAME").isin("Hollywood", "Wilshire", "West LA", "Pacific"), "SPA 5 - West")
    .when(col("AREA_NAME").isin("Van Nuys", "West Valley", "North Hollywood", "Foothill", "Devonshire", "Mission", "Topanga"), "SPA 2 - San Fernando")
    .when(col("AREA_NAME").isin("Northeast"), "SPA 4 - Metro") 
    .otherwise("Other")
)

In [98]:
# 4. Perform the Join
# We join on 'spa' to bring med_hh_income and ami_category into the crime records
final_merged_df = df_crime_with_spa.join(
    df_income.select("spa", "med_hh_income", "ami_category").distinct(), 
    on="spa", 
    how="left"
)

final_merged_df.printSchema()

root
 |-- spa: string (nullable = false)
 |-- DR_NO: integer (nullable = true)
 |-- Date_Rptd: string (nullable = true)
 |-- DATE_OCC: string (nullable = true)
 |-- TIME_OCC: integer (nullable = true)
 |-- AREA: integer (nullable = true)
 |-- AREA_NAME: string (nullable = true)
 |-- Rpt_Dist_No: integer (nullable = true)
 |-- Part_1-2: integer (nullable = true)
 |-- Crm_Cd: integer (nullable = true)
 |-- Crm_Cd_Desc: string (nullable = true)
 |-- Mocodes: string (nullable = true)
 |-- Vict_Age: integer (nullable = true)
 |-- Vict_Sex: string (nullable = true)
 |-- Vict_Descent: string (nullable = true)
 |-- Premis_Cd: integer (nullable = true)
 |-- Premis_Desc: string (nullable = true)
 |-- Weapon_Used_Cd: integer (nullable = true)
 |-- Weapon_Desc: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Status_Desc: string (nullable = true)
 |-- Crm_Cd_1: integer (nullable = true)
 |-- Crm_Cd_2: integer (nullable = true)
 |-- Crm_Cd_3: integer (nullable = true)
 |-- Crm_Cd

In [104]:
# 5. Save the merged data:
folder_path = "CS226_project"
# We use repartition(1) to get one single CSV file inside the folder
output_name = os.path.join(folder_path, "LA_Crime_Income_Merged_Final.csv")

final_merged_df.repartition(1).write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv(output_name)

print(f"Successfully saved combined data to: {output_name}")

Successfully saved combined data to: CS226_project/LA_Crime_Income_Merged_Final.csv
